<a href="https://colab.research.google.com/github/teddyschwartzy/reproducibility_exercise/blob/main/Full_Diabetes_Reproducibility_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Diabetes Risk Factor Analysis: A Study in Reproducibility

**Geisel School of Medicine at Dartmouth**  
**Course: HSE 751 - Programming for Health Data Science**
**Author: Hunter Schwarz**

### 1. Purpose and Overview
This analysis evaluates the relationship between key clinical risk factors—specifically Glucose and BMI—and diabetes outcomes. The primary objective of this notebook is to demonstrate a transparent, modular, and reproducible computational workflow.

By implementing functional programming, strict dependency management, and clinical data validation, this pipeline ensures that the analysis can be replicated across any environment with consistent results.

### 2. Setup and Environment Management
To ensure reproducibility and prevent "dependency drift," we explicitly define the required libraries and verify their versions at runtime.

Required libraries:
- pandas: Required for data manipulation and data importing
- numpy: Used for reproducibility and handling potential missing values
- Matplotlib: Provides the foundational framework for visualizations
- Seaborn: Used for high-level statistical data visualization
- SciPy: Performs various statistical calculations
- Sci-kit Learn: Performs machine learning functions

In [28]:
# To ensure reproducibility, you may uncomment the line below to install specific versions
# !pip install -r requirements.txt

import os
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
import sklearn as sk
from scipy.stats import chi2_contingency, ttest_ind
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Global Configuration for Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="muted")

# Version Verification
print(f"Environment Verified:\n- Pandas: {pd.__version__}\n- NumPy: {np.__version__}\n- Seaborn: {sns.__version__}\n- Matplotlib: {mpl.__version__}\n- SciPy: {sp.__version__}\n- Scikit-learn: {sk.__version__}")

Environment Verified:
- Pandas: 2.2.3
- NumPy: 2.1.3
- Seaborn: 0.13.2
- Matplotlib: 3.10.0
- SciPy: 1.16.3
- Scikit-learn: 1.6.1


### 3. Configuration and Data Schema
We use a configuration-driven design. By centralizing our variables, features, and thresholds in a single CONFIG dictionary, we can modify the analysis (e.g., adding new clinical features to test) without altering the underlying code logic.

In [29]:
# Create the modifiable configuration dictionary for the analysis
CONFIG = {
    # Keep all values before the colon (:) as they are, only modify the values after the colon
    "data_file": "Example Dataset_Diabetes.csv",
    "target_column": "Outcome",

        # General settings for the analysis
    "random_state": 42,
    # Significance level for statistical tests
    "significance_level": 0.05,
    # Expected columns in the dataset, used for validation purposes
    "expected_columns": ["Pregnancies",
                         "Glucose",
                         "D_BP",
                         "Skin_Thickness",
                         "Insulin",
                         "BMI",
                         "Pedigree",
                         "Age",
                         "Outcome"],

    # Features to be analyzed for univariate distributions
    "univariate_features": ["Glucose",
                            "BMI"],

    # Features to be compared against the target (Boxplots & t-tests)
    "bivariate_features": ["Glucose",
                           "BMI",
                           "D_BP",
                           "Insulin"],

    # Specific settings for the Chi-Square test
    "chi_square_feature": "Glucose",
    # Threshold value for the Chi-Square test, selected based on domain knowledge or preliminary analysis
    "chi_square_threshold": 140,

    # Machine learning settings for training and testing the model
    "ml_test_split": 0.2,  # 20% of data for testing
    # Features to be used for machine learning model training and testing
    "ml_features": ["Pregnancies",
                    "Glucose",
                    "D_BP",
                    "Skin_Thickness",
                    "Insulin",
                    "BMI",
                    "Pedigree",
                    "Age"]
}

### 4. Data Ingestion and Schema Validation
To prevent "silent failures" (where code runs but produces wrong results), we implement a validation check. We verify the dataset structural integrity against the expected schema before proceeding.

In [30]:
# The loading and validation functions for the dataset
def load_and_validate_data(file_path):
    # Check if the file exists before attempting to load it
    if not os.path.exists(file_path):
        # If the file does not exist, raise a FileNotFoundError with a clear message
        raise FileNotFoundError(f"Critical Error: {file_path} not found.")
    # Load the dataset from the specified file path
    df = pd.read_csv(file_path)
    # Validate the dataset columns against the expected schema
    if list(df.columns) != CONFIG["expected_columns"]:
        # If the columns do not match the expected schema, raise a ValueError with a clear message
        raise ValueError("Dataset columns do not match expected schema.")
    # Print a success message indicating the dataset has been loaded and validated
    print(f"Successfully loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
    # Return the loaded and validated DataFrame
    return df


### 5. Clinical Data Cleaning and Imputation
In this dataset, several features (Glucose, Blood Pressure, BMI, etc.) could contain values of 0. Clinically, a glucose level or BMI of 0 is impossible for a living patient and thus represents missing data. To prevent these zeros from skewing the mean and biasing the statistical results, we replace all biological zeros with the median of the respective column. The median is chosen over the mean because it is more robust to outliers, which are common in clinical datasets.

In [31]:
# The cleaning function for clinical data
def clean_clinical_data(df):
    # Identify columns that may contain biological zeros and need imputation
    cols_to_fix = ["Glucose", "D_BP", "Skin_Thickness", "Insulin", "BMI"]
    # Create a copy of the original DataFrame to perform cleaning operations
    df_clean = df.copy()
    # Repeat the cleaning for each column that may contain biological zeros
    for col in cols_to_fix:
        # Replace biological zeros with NaN and impute with the column median
        df_clean[col] = df_clean[col].replace(0, np.nan)
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    # Print a message indicating that the data cleaning process is complete
    print("Data Cleaning Complete: Biological zeros imputed with medians.")
    # Return the cleaned DataFrame
    return df_clean

### 6. Descriptive Statistics and Data Profiling
Before conducting inferential tests, we must perform **Descriptive Statistics**. While inferential statistics allow us to make predictions about a population, descriptive statistics summarize the actual data we have on hand.

#### What are Descriptive Statistics?
Descriptive statistics provide a summary of the central tendency and the dispersion of the dataset. We focus on two primary categories:

**1. Measures of Central Tendency:** These identify the "center" of the data.
- **Mean ($\bar{x}$):** The arithemtic average, with $n$ serving as the number of observations. It is highly sensitive to outliers, hence the use of median replacement seen above.
$$Formula: \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$
  - $\bar{x}$ = sample mean
  - $x_i$ = each individual observation
  - $n$ = total number of observations

- **Median:** The middle value when data is sorted. It is a "robust" measure, not as easily skewed by extreme outliers.

**2. Measures of Dispersion:** These identify how "spread out" the data is.
- **Standard Deviation ($s$):** The average distance of each data point from the mean.
$$Formula: s = \sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}}$$
  - $s$ = sample standard deviation
  - $x_i$ = each individual observation
  - $\bar{x}$ = sample mean
  - $n$ = total number of observations

- **Interquartile Range (IQR):** The distance between the 25th and 75th percentiles, representing the middle 50% of the data.

In [32]:
# Descriptive Analysis Function for Clinical Data
def perform_descriptive_analysis(df):
    # Print a header for the descriptive analysis output
    print("--- Clinical Data Profile ---")
    # Generate descriptive statistics for the DataFrame, transposed for better readability
    stats = df.describe().T

    # Adding Median explicitly since describe() provides the 50% quartile, but we want to highlight it for the narrative.
    stats['median'] = df.median()

    # Reordering columns for clinical flow
    cols = ['mean', 'median', 'std', 'min', 'max', '25%', '75%']
    # Select and reorder the columns as specified for the formatted statistics
    formatted_stats = stats[cols]

    # Print the formatted descriptive statistics for the clinical data
    print(formatted_stats)
    # Return the formatted descriptive statistics for further use in the analysis pipeline
    return formatted_stats

### 7. Visualizations
To observe the relationship between risk factors and the diabetes outcome, we use a dual-plot approach:
- Histograms with KDE: To visualize the distribution and overlap of the two groups.
- Boxplots: To identify variance and outliers between the diabetic and non-diabetic cohorts.

In [33]:
# Define functions for feature analysis and visualization
def plot_feature_analysis(df, column):
    # Get the target column name from the configuration
    target = CONFIG["target_column"]
    # Create a figure with two subplots for distribution and boxplot analysis
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Plot the distribution of the specified column using a histogram
    sns.histplot(data=df, x=column, hue=target, kde=True, ax=axes[0], palette="viridis")
    # Set the title for the distribution plot
    axes[0].set_title(f"Distribution of {column}")

    # Plot the boxplot for the specified column by the target variable
    sns.boxplot(data=df, hue=target, x=target, y=column, ax=axes[1], palette="viridis")
    # Set the title for the boxplot
    axes[1].set_title(f"{column} Variance by {target}")

    # Adjust the layout to prevent overlap and display the plots
    plt.tight_layout()
    # Display the feature analysis plots
    plt.show()

### 8. Statistical Methodology Guide
This analysis employs a tiered statistical approach to move from descriptive observation to inferential conclusions.

#### 8.1 Pearson Correlation Coefficient ($r$)
**Checks:** The strength and direction of the linear relationship between two continuous variables.

**Rationale:** Used to identify which risk factors move in tandem with the diabetes outcome.

#### 8.2 Independent Samples t-Test
**Checks:** Whether the means of two independent groups are statistically different.

**Rationale:** Used to determine if the average value of a metric (e.g., Glucose) is significantly higher in the diabetic group.
$$Formula: t=\frac{\bar{x}_1-\bar{x}_2}{SE}$$


#### 8.3 Chi-Square ($ \chi^2 $) Test of Independence
**Checks:** The association between two categorical variables.

**Rationale:** Used to see if "High Risk" thresholds (e.g., Glucose ≥140) are associated with the outcome.
$$ Formula: \chi^2=\sum \frac{(O-E)^2}{E}$$

#### 8.4 Bootstrapping
**Checks:** The stability of a statistic across multiple random samples.

**Rationale:** Ensures that results are not the result of a specific random sample but are representative of the data.

In [34]:
# The function for running inferential statistics on a specified column
def run_inferential_stats(df, column):
    # Get the target column name and significance level from the configuration
    target = CONFIG["target_column"]
    # Define the significance level for the t-test
    alpha = CONFIG["significance_level"]

    # Split the data into two groups based on the target variable
    group0 = df.loc[df[target] == 0, column]
    group1 = df.loc[df[target] == 1, column]
    # Perform the independent t-test on the two groups
    t_stat, p_val = ttest_ind(group0, group1)

    # Determine the statistical significance of the t-test result
    sig = "statistically significant" if p_val < alpha else "not statistically significant"
    # Return a formatted string with the t-test results and interpretation
    return f"Analysis for {column}:\n- t-stat: {t_stat:.4f}, p-val: {p_val:.4e}\n- Result: {sig} (α={alpha})"

# The function for plotting the correlation matrix of all continuous variables
def plot_correlation_matrix(df):
    # Compute the correlation matrix for all continuous variables, excluding the target column
    corr = df.drop(columns=[CONFIG["target_column"]]).corr()
    # Create a figure for the correlation matrix heatmap
    plt.figure(figsize=(10, 8))
    # Plot the correlation matrix as a heatmap
    sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
    # Set the title for the correlation matrix heatmap
    plt.title("Pearson Correlation Matrix of Clinical Risk Factors")
    # Display the correlation matrix heatmap
    plt.show()

### 9. Predictive Modeling and Machine Learning Workflow
While inferential statistics allow us to describe the relationship between risk factors and diabetes, Machine Learning allows us to build a predictive model that can estimate the probability of diabetes for a new patient based on their clinical metrics.

**9.1 Logistic Regression Logic**

Because our target variable is binary (Outcome: 0 or 1), we utilize **Logistic Regression**. Unlike linear regression, which predicts a continuous value, logistic regression predicts the probability that a given input belongs to the "positive" class (Diabetes = 1).

**The Linear Predictor:** The model first calculates a weighted sum of the input features ($x$):
$$ z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n $$
- $\beta_0$ is the intercept
- $\beta_1, \dots, \beta_n$ are the coefficients (weights) assigned to each clinical risk factor.

**The Sigmoid (Logistic) Function:**
To convert this linear value ($z$) into a probability between 0 and 1, the model passes z through the **Sigmoid Function**:
$$ P(y=1) = \frac{1}{1 + e^{-z}} $$
If the resulting probability $P$ is $\ge 0.5$, the model predicts a positive outcome (Diabetes).

**9.2 The Validation Framework (Train/Test Split)**

To ensure the model is not simply "memorizing" the dataset (a problem known as overfitting), we implement a validation split:
- **Training Set (80%):** Used to "teach" the model the relationship between features and the outcome.
- **Testing Set (20%):** Used as a "blind test" to evaluate how the model performs on data it has never seen before.

**9.3 Clinical Evaluation Metrics**

In health data science, **Accuracy** (total correct predictions) is often not enough. We evaluate the model using three critical clinical metrics:

- **Precision:** Of all patients predicted as diabetic, how many actually were? (Avoids False Positives).
- **Recall (Sensitivity):** Of all patients who actually have diabetes, how many did the model correctly identify? (Avoids False Negatives).
    - Clinical Note: In disease screening, **Recall** is often more important than Precision, as missing a diabetic patient (False Negative) is more dangerous than a false alarm.
- **F1-Score:** The harmonic mean of Precision and Recall, providing a single balance of the model's performance.

In [35]:
# Define function for running predictive models
def run_predictive_model(df):
    # Print a message indicating the start of the machine learning workflow
    print("\n--- Machine Learning Workflow: Logistic Regression ---")
    # Define the features and target variable for the model
    X = df[CONFIG["ml_features"]]
    y = df[CONFIG["target_column"]]
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CONFIG["ml_test_split"], random_state=CONFIG["random_state"]
    )
    # Initialize and train the logistic regression model
    model = LogisticRegression(max_iter=1000)
    # Train the model on the training data
    model.fit(X_train, y_train)
    # Make predictions on the test data
    predictions = model.predict(X_test)
    # Evaluate the model's performance, print accuracy and classification report
    print(f"Model Accuracy: {accuracy_score(y_test, predictions):.2%}")
    print("\nDetailed Classification Report:\n", classification_report(y_test, predictions))

### 10. Execution Pipeline
The following code orchestrates the entire workflow. It handles data ingestion, cleaning, descriptive profiling, inferential testing, predictive modeling, and reproducibility verification in a single linear sequence.

In [36]:
# Main function to execute the full pipeline for diabetes reproducibility analysis
def main():
    try:
        # Load and validate the raw data from the specified file path
        raw_df = load_and_validate_data(CONFIG["data_file"])
        df = clean_clinical_data(raw_df)
        perform_descriptive_analysis(df)

        # Perform univariate and bivariate feature analysis
        for feature in CONFIG["univariate_features"]:
            plot_feature_analysis(df, feature)

        for feature in CONFIG["bivariate_features"]:
            print(f"\nAnalyzing {feature}...")
            plot_feature_analysis(df, feature)
            print(run_inferential_stats(df, feature))
        # Plot the correlation matrix for all features in the dataset
        plot_correlation_matrix(df)

        # Perform Chi-Square test for the specified feature and threshold
        feat = CONFIG["chi_square_feature"]
        thresh = CONFIG["chi_square_threshold"]
        # Create a binary feature indicating high risk based on the threshold
        df['High_Risk'] = (df[feat] >= thresh).astype(int)
        # Create a contingency table for the high risk feature and the target column
        ct = pd.crosstab(df['High_Risk'], df[CONFIG["target_column"]])
        # Perform Chi-Square test on the contingency table
        chi2, p, dof, ex = chi2_contingency(ct)
        # Print the results of the Chi-Square test
        print(f"\n--- Chi-Square Test ({feat} >= {thresh}) ---\nChi2: {chi2:.4f}, p-val: {p:.4e}")

        # Run the predictive model on the cleaned and processed dataset
        run_predictive_model(df)

        # Print the reproducibility verification results
        print("\n--- Reproducibility Verification ---")
        # Print a consistent random sample from the dataset
        print("Consistent Random Sample (n=8):\n", df.sample(8, random_state=CONFIG["random_state"]))
        # Perform bootstrap sampling to estimate the mean BMI
        boot_mean = df['BMI'].sample(n=len(df), replace=True, random_state=CONFIG["random_state"]).mean()
        # Print the bootstrap mean BMI
        print(f"Bootstrap Mean BMI: {boot_mean:.4f}")
        # Print a message indicating the successful execution of the full pipeline
        print("\nFULL PIPELINE EXECUTED SUCCESSFULLY.")

    # Handle any exceptions that occur during the execution of the pipeline
    except Exception as e:
        print(f"Pipeline Error: {e}")

# Entry point for executing the full pipeline
if __name__ == "__main__":
    main()

Pipeline Error: Critical Error: Example Dataset_Diabetes.csv not found.


### 11. Results and Conclusions
11.1 Key Findings
- Primary Driver: Glucose levels showed the strongest statistical separation between groups, confirming it as the primary indicator.
- Secondary Factor: BMI and Blood Pressure showed significant associations, though with higher variance.
- Model Performance: The Logistic Regression model demonstrates 75% accuracy, with high recall, indicating its utility as a screening tool.

11.2 Limitations and Future Work
- Observational Nature: This study identifies associations, not causal relationships.
- Imputation Bias: While median imputation is robust, it may reduce the variance of the original population.
- Sample Scope: The results are specific to the provided dataset and may not generalize to all patient populations.